In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/Diego/Paper Interpretability')
print(os.getcwd())

In [ ]:
!pip install optuna
!pip install lime
!pip install tf-keras-vis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 25.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=b43f3e0b0c8d57a1d1b5f3df4ff42360f65639695a9eb3667a536c621df16d44
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
import sys
import platform
import subprocess
import importlib
import os

def get_pkg_version(pkg_name, import_name=None):
    import_name = import_name or pkg_name
    try:
        module = importlib.import_module(import_name)
        return getattr(module, "__version__", "version attribute not found")
    except Exception as e:
        return f"not installed or not importable ({type(e).__name__}: {e})"

def run_cmd(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
        return out.strip()
    except Exception as e:
        return f"command failed: {e}"

print("="*80)
print("SYSTEM")
print("="*80)
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Architecture:", platform.architecture())
print("Machine:", platform.machine())
print("Processor:", platform.processor())
print("OS:", run_cmd("cat /etc/os-release | grep PRETTY_NAME"))

print("\n" + "="*80)
print("CPU / RAM")
print("="*80)
print("CPU info:")
print(run_cmd("lscpu | grep -E 'Model name|CPU\\(s\\)|Thread|Core|Socket'"))
print("\nRAM:")
print(run_cmd("free -h"))

print("\n" + "="*80)
print("GPU / CUDA / NVIDIA DRIVER")
print("="*80)
print("nvidia-smi:")
print(run_cmd("nvidia-smi"))
print("\nCUDA nvcc:")
print(run_cmd("nvcc --version"))

print("\n" + "="*80)
print("COLAB GPU MEMORY")
print("="*80)
print(run_cmd("nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv"))

print("\n" + "="*80)
print("PYTHON LIBRARIES")
print("="*80)

packages = {
    "NumPy": ("numpy", "numpy"),
    "SciPy": ("scipy", "scipy"),
    "scikit-learn": ("sklearn", "sklearn"),
    "TensorFlow": ("tensorflow", "tensorflow"),
    "Keras": ("keras", "keras"),
    "KerasNLP": ("keras_nlp", "keras_nlp"),
    "SHAP": ("shap", "shap"),
    "LIME": ("lime", "lime"),
    "tf-keras-vis": ("tf_keras_vis", "tf_keras_vis"),
    "Optuna": ("optuna", "optuna"),
    "Pandas": ("pandas", "pandas"),
    "Matplotlib": ("matplotlib", "matplotlib"),
    "MNE": ("mne", "mne"),
}

for label, (_, import_name) in packages.items():
    print(f"{label}: {get_pkg_version(label, import_name)}")

print("\n" + "="*80)
print("TENSORFLOW BUILD INFO")
print("="*80)
try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
    print("Built with CUDA:", tf.test.is_built_with_cuda())
    print("GPUs visible to TensorFlow:", tf.config.list_physical_devices("GPU"))
    print("Build info:", tf.sysconfig.get_build_info())
except Exception as e:
    print("TensorFlow info error:", e)

print("\n" + "="*80)
print("PIP FREEZE SELECTED")
print("="*80)
selected = [
    "numpy", "scipy", "scikit-learn", "tensorflow", "keras", "keras-nlp",
    "shap", "lime", "tf-keras-vis", "optuna", "pandas", "matplotlib", "mne"
]
freeze = run_cmd("pip freeze")
for line in freeze.splitlines():
    if any(line.lower().startswith(pkg.lower() + "==") for pkg in selected):
        print(line)

SYSTEM
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Architecture: ('64bit', 'ELF')
Machine: x86_64
Processor: x86_64
OS: PRETTY_NAME="Ubuntu 22.04.5 LTS"

CPU / RAM
CPU info:
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
NUMA node0 CPU(s):                       0,1

RAM:
total        used        free      shared  buff/cache   available
Mem:            12Gi       1.0Gi       7.6Gi       2.0Mi       4.0Gi        11Gi
Swap:             0B          0B          0B

GPU / CUDA / NVIDIA DRIVER
nvidia-smi:
Wed May 20 05:47:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      C

In [ ]:
import os
import gc
import numpy as np
import tensorflow as tf
import shap

from copy import deepcopy
from lime.lime_tabular import LimeTabularExplainer

from Model_utils.model_builders import build_eeg_model

from Model_utils.utils import (
    infer_chans_samples_from_X,
    load_best_trial_from_journal,
    extract_probs_from_model_output,
)

from Model_utils.evaluation_pipeline import (
    get_model_paths,
    get_weights_path,
    split_best_params_for_training,
)
import os
import pickle

import numpy as np
from Model_utils.utils import get_segmented_data
from Model_utils.xai_runner import run_mi_xai_and_save,run_tdah_xai_and_save


In [ ]:
import os
import pickle
import numpy as np

# ============================================================
# MI data
# ============================================================

base_path_mi = "Data/MI"

mi_subjects_to_extract = {
    "2.5-5": {
        43: 1,
        12: 3,
    },
    "0-7": {
        14: 4,
        12: 5,
    },
}

mi_data = {}

for window_name, subjects_folds in mi_subjects_to_extract.items():
    mi_data[window_name] = {}

    for subject_id, fold in subjects_folds.items():
        file_path = os.path.join(
            base_path_mi,
            window_name,
            f"subject_{subject_id}.npz"
        )

        data = np.load(file_path)

        X = data["X"]
        y = data["y"]

        test_idx = data[f"fold_{fold}_test_idx"]

        train_idx = np.setdiff1d(
            np.arange(len(y)),
            test_idx
        )

        mi_data[window_name][subject_id] = {
            "fold": fold,

            "X_train": X[train_idx].astype(np.float32),
            "y_train": y[train_idx],

            "X_test": X[test_idx].astype(np.float32),
            "y_test": y[test_idx],

            "train_idx": train_idx,
            "test_idx": test_idx,
        }

        data.close()


# ============================================================
# TDAH data
# ============================================================

with open("Data/TDAH/folds.pkl", "rb") as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data(
    path_adhd="Data/TDAH/ieee/ADHD_group",
    path_control="Data/TDAH/ieee/Control_group",
)

fold_to_extract = 4

train_subjects, _, test_subjects = folds[fold_to_extract]

sbjs_array = np.array(sbjs)

train_idx = [
    i for i, sbj in enumerate(sbjs)
    if sbj in train_subjects
]

test_idx = [
    i for i, sbj in enumerate(sbjs)
    if sbj in test_subjects
]

train_idx = np.asarray(train_idx, dtype=int)
test_idx = np.asarray(test_idx, dtype=int)

tdah_data = {
    "fold": fold_to_extract,

    "X_train": X[train_idx].astype(np.float32),
    "y_train": y[train_idx],

    "X_test": X[test_idx].astype(np.float32),
    "y_test": y[test_idx],

    "sbjs_train": sbjs_array[train_idx],
    "sbjs_test": sbjs_array[test_idx],

    "train_idx": train_idx,
    "test_idx": test_idx,
}

In [ ]:
mi_subjects_to_extract = {
    "2.5-5": {
       43: 1,
       12: 3,
    },
     "0-7": {
         14: 4,
         12: 5,
     },
}


run_mi_xai_and_save(
     mi_data=mi_data,
     mi_subjects_to_extract=mi_subjects_to_extract,
     results_dir="Results",
     models_to_run=[
         "tgarnet",
         "shallowconvnet",
     ],
     xai_methods_to_run=[
          # "KernelSHAP",
          # "STF-KernelSHAP",
          # "LIME",
          # "Occlusion",
          # "IntegratedGradients",
          "GradCAM++",
     ],
     selected_windows=None,
     selected_subjects=None,
     overwrite=True,
     use_y_test=True,
     predict_batch_size=32,
 )


##########################################################################################
Ventana: 2.5-5 | Sujeto: 43 | Fold: 1
##########################################################################################
X_test shape: (40, 64, 320)
N muestras: 40

Cargando modelo: tgarnet
Etiquetas usadas para XAI: y_test

--------------------------------------------------------------------------------
Modelo: tgarnet | Método: GradCAM++ | Ventana: 2.5-5 | Sujeto: 43
--------------------------------------------------------------------------------
Parámetros XAI: {'layer_name': None}
[1/40] GradCAM++ | sample_idx=0 | class=0 | layer=Conv2D_2 | logits
[2/40] GradCAM++ | sample_idx=1 | class=0 | layer=Conv2D_2 | logits
[3/40] GradCAM++ | sample_idx=2 | class=0 | layer=Conv2D_2 | logits
[4/40] GradCAM++ | sample_idx=3 | class=0 | layer=Conv2D_2 | logits
[5/40] GradCAM++ | sample_idx=4 | class=0 | layer=Conv2D_2 | logits
[6/40] GradCAM++ | sample_idx=5 | class=1 | layer=Conv2D_2 | logits
[7

0-7

In [ ]:
fold_to_extract = 5

tdah_data_by_fold = {
   fold_to_extract: tdah_data,
}

tdah_folds_to_extract = [
   fold_to_extract,
]


run_tdah_xai_and_save(
   tdah_data_by_fold=tdah_data_by_fold,
   folds_to_extract=tdah_folds_to_extract,
   results_dir="Results",
   models_to_run=[
       "shallowconvnet",
       "tgarnet",
   ],
  xai_methods_to_run=[
       #"KernelSHAP",
       #"STF-KernelSHAP",
       # "LIME",
        # "Occlusion",
        # "IntegratedGradients",
         "GradCAM++",
   ],
   selected_folds=None,
   overwrite=True,
   use_y_test=True,
   predict_batch_size=32,
   n_partitions=None,#7,
   partition_id=None,#7,
)


##########################################################################################
TDAH | Fold: 5
##########################################################################################
X_test original shape: (1722, 19, 512)
N muestras originales: 1722
Partición usada: ninguna, se usa todo X_test.
X_test usado shape: (1722, 19, 512)
N muestras usadas: 1722

Cargando modelo TDAH: shallowconvnet
Etiquetas usadas para XAI: y_test

--------------------------------------------------------------------------------
TDAH | Fold: 5 | Modelo: shallowconvnet | Método: GradCAM++
--------------------------------------------------------------------------------
Parámetros XAI: {'layer_name': 'Conv2D_1'}
[1/1722] GradCAM++ | sample_idx=0 | class=0 | layer=Conv2D_1 | logits
[2/1722] GradCAM++ | sample_idx=1 | class=0 | layer=Conv2D_1 | logits
[3/1722] GradCAM++ | sample_idx=2 | class=0 | layer=Conv2D_1 | logits
[4/1722] GradCAM++ | sample_idx=3 | class=0 | layer=Conv2D_1 | logits
[5/1722] Gr